# Phase 3-4: Optimized GPU with BCE Loss + SVM Integration (Colab Version)
**CSC14120 - Parallel Programming**

---

## Phase 3: BCE Loss with Sigmoid

### Architecture Changes:
- **Output Activation:** Sigmoid after Conv5
- **Loss Function:** Binary Cross-Entropy (BCE) instead of MSE

### Why BCE + Sigmoid:

| Aspect | MSE | BCE + Sigmoid |
|:-------|:----|:--------------|
| Output range | Unbounded (-∞, +∞) | Bounded [0, 1] |
| Gradient | Linear | Stronger near 0.5 |
| Pixel validity | Needs clipping | Always valid |
| Theory | Regression | Probabilistic |

---

## Phase 4: SVM Classification

### Pipeline:
1. Extract features using trained encoder (8×8×128 = 8192 dims)
2. Preprocess with StandardScaler
3. Train SVM classifier (RBF kernel)
4. Evaluate on CIFAR-10 test set

### Target: 60-65% accuracy

---

## Instructions:
1. Upload project zip (without data/)
2. Run all cells
3. Download results

In [ ]:
# Check GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Upload and extract project
from google.colab import files
import zipfile, os

print('Upload project zip file:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

# Find repo root (folder containing Makefile + src/ + include/)
project_root = None
for root, dirs, files_ in os.walk('project'):
    if 'src' in dirs and 'include' in dirs and 'Makefile' in files_:
        project_root = root
        break

assert project_root is not None, 'Cannot find project root containing Makefile + src/ + include/'
%cd {project_root}

print('Project root:', os.getcwd())
!ls -la

In [ ]:
# Download CIFAR-10 dataset
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/test_batch.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz',
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
    !rm -rf data/cifar-10-batches-bin data/cifar.tar.gz

print('CIFAR-10 files:')
!ls -la data/*.bin

---
# Phase 3: Training with BCE Loss

In [ ]:
# Build Phase 3 with cuDNN (BCE is runtime flag)
# Note: Colab typically uses sm_75 for Tesla T4
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -Iinclude -lcublas -lcudnn -lcurand \
    -o gpu_train_bce \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build completed!')

## Training with BCE Loss

**Important Notes:**
- BCE loss values are typically **higher** than MSE loss (different scale)
- Don't compare BCE loss directly with MSE loss
- Evaluate via reconstruction quality and downstream accuracy

In [ ]:
!./gpu_train_bce --data data --epochs 20 --batch 64 --lr 0.001 \
    --bce-loss \
    --log phase3_bce.csv --log-txt phase3_bce.txt --save-weights phase3_bce.weights

## Results and Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase3_bce.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep['epoch'], ep['loss'], 'purple', marker='o')
ax1.set_title('BCE Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.grid(True)

ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'green', marker='o')
ax2.set_title('Time per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Time (s)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('phase3_bce_results.png', dpi=150)
plt.show()

print("="*50)
print("PHASE 3 BCE LOSS RESULTS")
print("="*50)
print(f"Best BCE Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Final BCE Loss: {ep['loss'].iloc[-1]:.6f}")
print(f"Avg Time per Epoch: {ep['epoch_time_sec'].mean():.2f}s")
print(f"Total Training Time: {ep['epoch_time_sec'].sum():.2f}s ({ep['epoch_time_sec'].sum()/60:.2f} min)")
print("="*50)

In [ ]:
# Show training log
print("="*60)
print("TRAINING LOG (last 30 lines)")
print("="*60)
!tail -30 phase3_bce.txt

---
# Phase 4: SVM Integration

## Overview

### Feature Extraction
- Use trained encoder to extract 8192-dim features (8×8×128)
- GPU-accelerated forward pass
- Save to binary files for efficient loading

### SVM Training
- Preprocess with StandardScaler
- RBF kernel with tuned hyperparameters
- Target: 60-65% accuracy on CIFAR-10

In [ ]:
import time
import numpy as np
import struct
import gc

# Verify weights file exists
weights_file = "phase3_bce.weights"
if not os.path.exists(weights_file):
    print("ERROR: Weights file not found! Train Phase 3 first.")
else:
    print(f"Found weights: {weights_file}")
    !ls -lh {weights_file}

## Build Feature Extractor

In [ ]:
print("Building LIBSVM...")
!git clone --depth 1 https://github.com/cjlin1/libsvm.git libsvm_src 2>/dev/null || echo "Already cloned"
!cd libsvm_src && make lib
!cp libsvm_src/svm.h include/ 2>/dev/null || true
!cp libsvm_src/svm.cpp src/ 2>/dev/null || true

print("\nBuilding feature_extractor...")
!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DWITH_SVM -DWITH_LIBSVM \
    -Iinclude -Ilibsvm_src -lcublas -lcudnn \
    -o feature_extractor \
    src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu \
    src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp libsvm_src/svm.cpp

!if [ -f feature_extractor ]; then echo "Build SUCCESS!"; else echo "Build FAILED!"; fi

## Extract Features

In [ ]:
print("Extracting features with GPU autoencoder...")
start = time.time()
!./feature_extractor --data data --weights phase3_bce.weights --extract-only
print(f"\nFeature extraction: {time.time() - start:.2f}s")
!ls -lh *.bin 2>/dev/null || echo "No .bin files"

In [ ]:
# Load features and labels
feature_dim = 8192

print("Loading features...")
train_features = np.fromfile('train_features.bin', dtype=np.float32).reshape(-1, feature_dim)
test_features = np.fromfile('test_features.bin', dtype=np.float32).reshape(-1, feature_dim)

print(f"Train features: {train_features.shape}")
print(f"Test features: {test_features.shape}")

# Load labels
def load_cifar10_labels(data_dir):
    train_labels, test_labels = [], []
    for i in range(1, 6):
        with open(f"{data_dir}/data_batch_{i}.bin", 'rb') as f:
            for _ in range(10000):
                train_labels.append(struct.unpack('B', f.read(1))[0])
                f.read(3072)
    with open(f"{data_dir}/test_batch.bin", 'rb') as f:
        for _ in range(10000):
            test_labels.append(struct.unpack('B', f.read(1))[0])
            f.read(3072)
    return np.array(train_labels), np.array(test_labels)

train_labels, test_labels = load_cifar10_labels('data')
print(f"Labels loaded: {len(train_labels)} train, {len(test_labels)} test")

## Feature Preprocessing

Using StandardScaler to normalize features (zero mean, unit variance)

In [ ]:
from sklearn.preprocessing import StandardScaler

print("="*60)
print("FEATURE PREPROCESSING")
print("="*60)

print("\nStandardizing features...")
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features)

print(f"Train mean: {train_features_scaled.mean():.6f}, std: {train_features_scaled.std():.6f}")
print(f"Feature dim: {train_features_scaled.shape[1]}")

# Use scaled features
X_train = train_features_scaled
X_test = test_features_scaled
y_train = train_labels
y_test = test_labels

# Free memory
del train_features, test_features, train_features_scaled, test_features_scaled, scaler
gc.collect()
print("\nMemory freed!")

## Hyperparameter Tuning

Quick search on subset to find best C parameter

In [ ]:
from sklearn.svm import SVC

# Use small subset for quick tuning
tune_size = 10000
tune_indices = np.random.choice(len(X_train), size=tune_size, replace=False)
X_tune = X_train[tune_indices]
y_tune = y_train[tune_indices]

print(f"Quick hyperparameter search on {tune_size} samples...")
print("="*60)

# Test different C values
configs = [
    {'kernel': 'rbf', 'C': 8, 'gamma': 'scale', 'random_state': 42},
    {'kernel': 'rbf', 'C': 9, 'gamma': 'scale', 'random_state': 42},
    {'kernel': 'rbf', 'C': 10, 'gamma': 'scale', 'random_state': 42},
    {'kernel': 'rbf', 'C': 11, 'gamma': 'scale', 'random_state': 42},
    {'kernel': 'rbf', 'C': 12, 'gamma': 'scale', 'random_state': 42},
]

results = []
for cfg in configs:
    svm = SVC(**cfg)
    
    start = time.time()
    # Simple train/test split for quick eval
    split_idx = int(len(X_tune) * 0.8)
    svm.fit(X_tune[:split_idx], y_tune[:split_idx])
    preds = svm.predict(X_tune[split_idx:])
    acc = (preds == y_tune[split_idx:]).mean()
    elapsed = time.time() - start
    
    result = {
        'config': cfg,
        'accuracy': float(acc),
        'time': elapsed
    }
    results.append(result)
    print(f"{cfg}")
    print(f"   -> Accuracy: {acc*100:.2f}%, Time: {elapsed:.1f}s\n")
    
    del svm
    gc.collect()

# Find best config
best = max(results, key=lambda x: x['accuracy'])
print(f"\nBest config: {best['config']}")
print(f"Expected accuracy: {best['accuracy']*100:.2f}%")

del X_tune, y_tune
gc.collect()

## Train Final SVM

In [ ]:
print("="*60)
print("TRAINING FINAL SVM")
print("="*60)

# Use best parameters from tuning
KERNEL = best['config']['kernel']
C = best['config']['C']
GAMMA = best['config']['gamma']

print(f"\nParameters:")
print(f"  Kernel: {KERNEL}")
print(f"  C: {C}")
print(f"  gamma: {GAMMA}")
print(f"  Training samples: {len(X_train)}")
print(f"  Feature dim: {X_train.shape[1]}")

# Train SVM
svm = SVC(kernel=KERNEL, C=C, gamma=GAMMA, random_state=42, verbose=True)

print(f"\nTraining SVM...")
start = time.time()
svm.fit(X_train, y_train)
train_time = time.time() - start

print(f"\nTraining completed in {train_time:.2f}s")

## Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

print("Evaluating on test set...")
y_pred = svm.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print("="*60)
print("PHASE 4 FINAL RESULTS")
print("="*60)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Training Time: {train_time:.2f}s")
print("="*60)

# Classification report
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix (Accuracy: {test_acc*100:.2f}%)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('phase4_confusion_matrix.png', dpi=150)
plt.show()

## Download Results

In [ ]:
from google.colab import files

# Download Phase 3 results
files.download('phase3_bce.csv')
files.download('phase3_bce.txt')
files.download('phase3_bce.weights')
files.download('phase3_bce_results.png')

# Download Phase 4 results
files.download('phase4_confusion_matrix.png')

print("All results downloaded!")